---
title: Fixed-horizon labels
execute:
  enabled: true
---

`q.label.fixed_horizon` measures the return at one future horizon. Returns above the positive threshold receive label 1, returns below the negative threshold receive -1, and observations inside the neutral band receive 0.

The threshold may be scalar or event-specific. This example labels volatility-scaled CUSUM events after 10 trading observations.

In [1]:
import pandas as pd

import qrt as q

close = q.data.datasets.load("spy")["close"]
close = close.loc[close.index.max() - pd.DateOffset(years=5) :]
volatility = close.pct_change(fill_method=None).ewm(span=20, min_periods=20).std()
events = q.label.cusum_filter(close, volatility.mul(0.5).where(volatility.gt(0)))
labels = q.label.fixed_horizon(
    close,
    10,
    events=events,
    threshold=volatility.mul(0.5),
)
labels.tail()

,end_time,return,threshold,label
event_time,,,,
2026-06-23,2026-07-08,0.016113,0.005554,1
2026-06-26,2026-07-13,0.027682,0.004893,1
2026-06-29,2026-07-14,0.014615,0.005370,1
2026-06-30,2026-07-15,0.010766,0.005229,1
2026-07-06,2026-07-20,-0.012232,0.004683,-1


## Inspect class balance

The output retains `end_time`, realized `return`, and the event-time threshold. Keep `event_time` and `end_time` when constructing validation folds so overlapping outcomes can be purged.

In [2]:
labels.groupby("label", observed=True).agg(
    observations=("label", "size"),
    average_return=("return", "mean"),
    average_threshold=("threshold", "mean"),
)

,observations,average_return,average_threshold
label,,,
-1,252,-0.029042,0.004836
0,91,-0.000487,0.005234
1,440,0.026843,0.004970


## Visualize the labels

The marker direction and color encode the class at each event. Shaded intervals show the future window used to compute each outcome; overlapping bands make dependence between nearby labels visible.

In [3]:
chart_start = close.index.max() - pd.DateOffset(years=1)
figure = q.plot.labels(
    close.loc[chart_start:],
    labels.loc[chart_start:],
    title="SPY fixed-horizon labels (latest year)",
)
figure.show(renderer="notebook_connected")